# 06. Linear SVM: Training-Only Model Selection

This notebook trains, tunes and evaluates a **linear Support Vector Machine (SVM)** for
spoiler detection.

* only the saved **training split** is used for every decision; the **test split** is used
  once, in section 10, after everything is frozen;
* folds come from `generate_tfidf_folds`, so TF-IDF and other encoders are fitted on training
  rows only (no leakage);
* metrics: spoiler-class precision, recall, F1, macro-F1 and average precision.

**What is SVM-specific here**
* the kernel-vs-linear experiment (section 3);
* the hinge vs squared-hinge loss choice (section 1);
* margin diagnostics: margin width, share of rows inside the margin (section 5);
* the decision threshold is **0** (the boundary itself), not a 0.5 probability;
* class imbalance is handled with **class weights** by default instead of oversampling (section 1).

## 1. Setup and Search Settings

**Stages and data size.** Kernel check, initial tuning and backward elimination use a
10,000-review random sample of the training split (`sample_size`).
Sections 8–9 always use every training row. Set `sample_size = None` to use full training partition.

**Experiment switches** (the same notebook runs the research-question variants):
* `group_by = "review_text"` (standard split) or `"movie_id"` (RQ3, unseen movies).

**Class imbalance.** Only ~26% of reviews are spoilers. `imbalance = "class_weight"` makes each
spoiler mistake cost about 2.8× more (`class_weight="balanced"`). `"oversample"` copies spoiler
rows instead. Use one, never both. On the review-text-only runs, the two gave the same CV F1 (0.5576 vs 0.5570, fold std ≈ 0.002),
so class weights are the default: same result, no duplicated rows, faster and deterministic.

**SVM hyperparameter grid**
* `C`: the soft-margin trade-off, searched on a log scale.
* `loss`: `"hinge"` is the textbook SVM loss; `"squared_hinge"` (LinearSVC's default) squares
  it, punishing large violations more. Both are valid SVMs; CV decides.
* `penalty`: `"l2"` is the usual SVM. `"l1"` sets unhelpful weights to exactly zero, so the
  model selects its own words. L1 only works with squared hinge.

**TF-IDF grid.** Same as logistic regression: unigrams vs unigrams + bigrams, `min_df=3`,
at most 5,000 terms per text field, so differences come from the model, not the features.

**Selection metric.** Mean validation **macro-F1**.
Spoiler-class F1 is recorded too.

`max_drop = 0.005` is the elimination tolerance (section 6), fixed *before* looking at results.

In [ ]:
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "src").is_dir(), "Start in the repository root or notebooks/."
sys.path.insert(0, str(PROJECT_ROOT))

from src.prepare_features import INITIAL_FEATURES, prepare_features
from src.split_data import RANDOM_STATE
from src.linear_svm import (
    tune_linear_svm, backward_elimination, refit_linear_svm, compare_kernels,
    top_features, block_importance,
)

# ---- data size and experiment switches ----
sample_size = 10000            # sections 3-7 only; sections 8-9 always use all training rows
k = 5
group_by = "review_text"       # "movie_id" for RQ3 (unseen movies)
remove_spoilers = False        # True for RQ4 (spoiler-warning ablation)
imbalance = "class_weight"     # "class_weight", "oversample" or "none"
selection_metric = "macro_f1"
initial_features = INITIAL_FEATURES.copy()
max_drop = 0.005
max_iter = 5000
tol = 1e-4

# ---- SVM grid (L1 + hinge is not a valid combination) ----
model_grid = [
    {"penalty": ["l2"], "loss": ["squared_hinge", "hinge"],
     "C": [0.001, 0.01, 0.03, 0.1, 0.3, 1.0]},
    {"penalty": ["l1"], "loss": ["squared_hinge"], "C": [0.01, 0.1, 1.0]},
]
full_model_grid = model_grid   # narrow this if full-training retuning (section 8) is too slow

# ---- TF-IDF grid ----
tfidf_grid = {"ngram_range": [(1, 1), (1, 2)], "min_df": [3], "max_features": [5000]}

# ---- kernel check (section 3) ----
run_kernel_check = True
kernel_sample_sizes = (2000, 4000, 8000)
kernel_C_values = (0.1, 1.0, 10.0)

experiment = f"linear_svm_{group_by}" + ("_ablated" if remove_spoilers else "")
print("Experiment:", experiment)

## 2. Load Only Training Data

Read the saved training split in chunks and join the movie details with `prepare_features`.
The sample is drawn uniformly without replacement *before* grouped CV, using the shared seed.
The test split is not opened here.

In [ ]:
split_dir = PROJECT_ROOT / "data/splits"
if group_by == "movie_id":
    split_dir = split_dir / "movie_disjoint"
train_path = split_dir / "IMDB_reviews_train.json"
movies_path = PROJECT_ROOT / "data/cleaned/IMDB_movie_details.json"
if not train_path.exists() or not movies_path.exists():
    raise FileNotFoundError(
        "Run download_data.py, data_cleaning.py and split_data.py first "
        "(add --group-by movie_id for the movie-disjoint split)."
    )

with train_path.open(encoding="utf-8") as source:
    total_training_rows = sum(1 for _ in source)
selected_rows = None
if sample_size is not None and sample_size < total_training_rows:
    selected_rows = np.random.default_rng(RANDOM_STATE).choice(
        total_training_rows, size=sample_size, replace=False,
    )

movies = pd.read_json(movies_path, lines=True, convert_dates=False)


def load_training(rows=None, columns=None):
    # Prepared features + labels + movie IDs for the chosen rows (all rows if None).
    chunks = []
    with pd.read_json(train_path, lines=True, convert_dates=False, chunksize=10000) as reader:
        for chunk in reader:
            if rows is not None:
                chunk = chunk.loc[chunk.index.isin(rows)]
            prepared = prepare_features(chunk, movies)
            prepared["is_spoiler"] = chunk["is_spoiler"].to_numpy()
            prepared["movie_id"] = chunk["movie_id"].to_numpy()
            prepared["source_row"] = chunk.index.to_numpy()
            chunks.append(prepared if columns is None else prepared[columns])
    return pd.concat(chunks, ignore_index=True)


model_data = load_training(selected_rows)
source_row_indices = model_data.pop("source_row").to_numpy()
print(f"Using {len(model_data):,} of {total_training_rows:,} training reviews.")
print(f"Spoiler prevalence: {model_data['is_spoiler'].mean():.2%}")

## 3. Kernel Check: Linear vs RBF SVM

On random training samples of increasing size, take the **first grouped
CV fold** and fit three models on identical matrices:

* `LinearSVC`: the model used in this notebook (primal solver, no kernel);
* `SVC(kernel="linear")`: the same kind of boundary, but solved through the kernel (dual) method;
* `SVC(kernel="rbf")`: a non-linear kernel SVM.

Each is tried at a few C values; the plot shows the best C per model and size. What to look for:
1. **Accuracy:** does the RBF kernel beat the linear models? If not, non-linearity isn't helping.
2. **Fit time vs sample size:** kernel SVMs grow much faster than linearly. Extrapolate to
   459,000 rows.
3. **Support vectors:** for a kernel SVM, every support vector is kept and compared with each
   new review at prediction time. A large share means slow predictions as well.

This uses training data only and is a *design check*; its scores are not reported as results.
Set `run_kernel_check = False` to skip it.

**Why not a kernel (e.g. RBF) SVM?** A kernel SVM works with the *dual form*, which compares
every pair of training reviews through $K(x_i, x_j)$. Three reasons make that a poor fit here:

1. **Cost.** Training time grows roughly between $n^2$ and $n^3$ in the number of reviews $n$,
   and memory grows with $n^2$. With $n \approx 459{,}000$ training reviews, that is
   infeasible on a laptop. A linear SVM (`LinearSVC`, solved with liblinear) scales roughly
   linearly in the number of non-zero TF-IDF entries.
2. **Not needed.** TF-IDF already maps each review into a space with tens of thousands of
   dimensions (one per word or phrase). Kernels exist to lift data into a higher-dimensional
   space where it becomes separable; here, the data is already high-dimensional, and text
   classes are typically close to linearly separable in that space.
3. **Interpretability.** A linear SVM has one weight per word, so we can read which words push a
   review towards "spoiler". This is needed for RQ1, RQ3 and RQ4. An RBF SVM has no such
   per-word weights.

In [ ]:
if run_kernel_check:
    kernel_tfidf = {name: values[0] for name, values in tfidf_grid.items()}
    kernel_results = compare_kernels(
        model_data, feature_columns=initial_features, sample_sizes=kernel_sample_sizes,
        C_values=kernel_C_values, tfidf_params=kernel_tfidf, k=k, group_by=group_by,
        random_state=RANDOM_STATE, imbalance=imbalance,
    )
    best_per_model = (kernel_results.sort_values("macro_f1", ascending=False)
                      .drop_duplicates(["sample_size", "model"])
                      .sort_values(["model", "sample_size"]))
    best_per_model["support_vector_share"] = (best_per_model["support_vectors"]
                                              / best_per_model["train_rows"])
    display(best_per_model[["model", "sample_size", "C", "macro_f1", "f1", "average_precision",
                            "fit_seconds", "predict_seconds", "support_vector_share"]])

    fig_kernel, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for name, rows in best_per_model.groupby("model"):
        axes[0].plot(rows["sample_size"], rows["macro_f1"], marker="o", label=name)
        axes[1].plot(rows["sample_size"], rows["fit_seconds"], marker="o", label=name)
    axes[0].set(xlabel="Training sample size", ylabel="Validation macro-F1 (best C)",
                title="Does a kernel improve accuracy?")
    axes[1].set(xlabel="Training sample size", ylabel="Fit time (seconds, log scale)",
                yscale="log", title="How does training time grow?")
    for ax in axes:
        ax.grid(alpha=0.25)
        ax.legend()
    fig_kernel.tight_layout()
    plt.show()

    # Rough extrapolation of fit time to the full training split (time ~ n^slope).
    extrapolation = []
    for name, rows in best_per_model.groupby("model"):
        if len(rows) >= 2 and (rows["fit_seconds"] > 0).all():
            slope, intercept = np.polyfit(np.log(rows["sample_size"]), np.log(rows["fit_seconds"]), 1)
            extrapolation.append({"model": name, "growth_exponent": slope,
                                  "estimated_full_fit_hours": np.exp(intercept)
                                  * total_training_rows ** slope / 3600})
    display(pd.DataFrame(extrapolation))

## 4. Initial Tuning With All Features

Every candidate (SVM settings × TF-IDF settings) uses the same grouped folds. For each TF-IDF
setting, encoders are fitted on each fold's training rows only and reused across SVM candidates.

Reported per candidate (means/std over the 5 **validation** folds):
* `macro_f1`, `f1`, `precision`, `recall`, `average_precision`: validation scores at threshold 0;
* `train_macro_f1`: the same score on the fold's training rows. A large gap between training and
  validation means overfitting;
* `margin_width`, `inside_margin`, `nonzero_weights`: SVM diagnostics (explained in section 5);
* `converged`: whether the solver finished within `max_iter` in every fold. Unconverged
  candidates stay visible but cannot be selected.

In [ ]:
baseline_summary, baseline_fold_scores, baseline_config = tune_linear_svm(
    model_data, model_grid=model_grid, tfidf_grid=tfidf_grid,
    feature_columns=initial_features, k=k, group_by=group_by, random_state=RANDOM_STATE,
    imbalance=imbalance, remove_spoilers=remove_spoilers, selection_metric=selection_metric,
    max_iter=max_iter, tol=tol,
)
if baseline_config is None:
    raise RuntimeError("No candidate converged in every fold. Increase max_iter and rerun.")
show_columns = ["penalty", "loss", "C", "ngram_range", "macro_f1_mean", "macro_f1_std",
                "f1_mean", "precision_mean", "recall_mean", "average_precision_mean",
                "train_macro_f1_mean", "inside_margin", "nonzero_weights", "seconds", "converged"]
display(baseline_summary[show_columns])
print("Best all-feature configuration:", baseline_config)

## 5. Inspect C: Bias–Variance and the Margin

Hold the best loss, penalty and TF-IDF setting fixed and look at how C changes things.

**Left: training vs validation macro-F1.**
* Small C: both scores low and close together → **underfitting** (margin too wide, too simple).
* Large C: training score keeps rising while validation flattens or drops → **overfitting**.
* The best C sits where validation peaks. If that is at the edge of the grid, extend the grid.

**Right: margin diagnostics.**
* *Margin width* $2/\lVert w\rVert$ shrinks as C grows: the model accepts a narrower street
  to fit more training reviews.
* *Share of training rows inside the margin* ($y\,f(x) < 1$): these rows produce hinge loss,
  and for a hinge-loss SVM they are exactly the **support vectors**. If most rows are inside the
  margin even at the best C, the two classes overlap heavily in this feature space. That is
  expected for spoilers, where many reviews are ambiguous, and explains why F1 is far from 1.

In [ ]:
best_model = baseline_config["model"]
same_setting = baseline_summary[
    (baseline_summary["penalty"] == best_model["penalty"])
    & (baseline_summary["loss"] == best_model["loss"])
    & baseline_summary["ngram_range"].map(lambda value: value == baseline_config["tfidf"]["ngram_range"])
].sort_values("C")

fig_c, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for column, label in [("train_macro_f1", "Training"), ("macro_f1", "Validation")]:
    axes[0].errorbar(same_setting["C"], same_setting[f"{column}_mean"],
                     yerr=same_setting[f"{column}_std"], marker="o", capsize=3, label=label)
axes[0].axvline(best_model["C"], color="grey", linestyle=":", label=f"Chosen C = {best_model['C']}")
axes[0].set(xscale="log", xlabel="C (log scale)", ylabel="Macro-F1",
            title=f"{best_model['penalty'].upper()} / {best_model['loss']}: training vs validation")
axes[0].legend()
axes[1].plot(same_setting["C"], same_setting["inside_margin"], marker="o", color="tab:purple",
             label="Share of training rows inside margin")
axes[1].set(xscale="log", xlabel="C (log scale)", ylabel="Share of training rows", ylim=(0, 1.05))
twin = axes[1].twinx()
twin.plot(same_setting["C"], same_setting["margin_width"], marker="s", color="tab:orange",
          label="Margin width 2/||w||")
twin.set(ylabel="Margin width", yscale="log")
axes[1].set_title("Margin vs C")
lines = axes[1].get_legend_handles_labels()[0] + twin.get_legend_handles_labels()[0]
axes[1].legend(lines, [line.get_label() for line in lines], loc="center right")
for ax in axes:
    ax.grid(alpha=0.25)
fig_c.tight_layout()
plt.show()

# Hinge vs squared hinge, and L1 vs L2, at their own best C.
display(baseline_summary.sort_values("macro_f1_mean", ascending=False)
        .drop_duplicates(["penalty", "loss", "ngram_range"])
        [["penalty", "loss", "ngram_range", "C", "macro_f1_mean", "f1_mean", "nonzero_weights"]])

## 6. Backward Feature Elimination

Freeze the SVM and TF-IDF settings from section 4. At each step, try removing each remaining
block once, keep the removal with the best mean validation score, and continue down to one
block. Then choose the **smallest** subset whose score is within `max_drop` (0.5 percentage
points) of the best score on the whole path. The tolerance is measured from the best score, not
per step, so small losses can't accumulate. This is a greedy search: it does not test every
possible subset. Folds stay identical at every step.

In [ ]:
selected_features, elimination_path, elimination_trials = backward_elimination(
    model_data, baseline_config, feature_columns=initial_features, max_drop=max_drop,
    selection_metric=selection_metric, k=k, group_by=group_by, random_state=RANDOM_STATE,
    imbalance=imbalance, remove_spoilers=remove_spoilers, max_iter=max_iter, tol=tol,
)
print("Selected features:", selected_features)
print("Removed features:", [f for f in initial_features if f not in selected_features])
display(elimination_path)

## 7. Elimination Plot

The line follows the best removal at each step. The shaded band is ±1 fold standard deviation
(not a confidence interval). The dashed line is the allowed floor; the red point is the
smallest acceptable subset.

In [ ]:
fig_elim, ax = plt.subplots(figsize=(12, 5.5))
steps = elimination_path["step"].to_numpy()
means = elimination_path["score_mean"].to_numpy()
stds = elimination_path["score_std"].to_numpy()
chosen = elimination_path.loc[elimination_path["selected"]].iloc[0]
ax.plot(steps, means, marker="o", label="Greedy elimination path")
ax.fill_between(steps, means - stds, means + stds, alpha=0.15, label="±1 fold std")
ax.axhline(means.max() - max_drop, color="tab:green", linestyle="--", label="Allowed floor")
ax.scatter(chosen["step"], chosen["score_mean"], color="tab:red", s=130, zorder=3,
           label="Selected subset")
ax.set_xticks(steps, [
    f"{row.n_features} blocks\n" + ("all" if row.step == 0 else f"- {row.removed_feature}")
    for row in elimination_path.itertuples()
], rotation=30, ha="right")
ax.set(ylabel=f"Mean validation {selection_metric}", title="Linear SVM: backward feature elimination")
ax.grid(axis="y", alpha=0.25)
ax.legend()
fig_elim.tight_layout()
plt.show()

## 8. Retune the Selected Features on the Full Training Split

Load **every** training row (still not the test split), keeping only the selected inputs plus
the columns needed for grouping and labels. Rerun the grid with grouped CV. The folds differ
from the sample's, so the comparison table states each stage's data scope; the score changes
are not a paired estimate of the elimination's effect.

This is the slowest step. If it takes too long, narrow `full_model_grid` in section 1 to the
region around the best C from section 4 (e.g. three C values and the best loss/penalty).

In [ ]:
if len(model_data) == total_training_rows:
    full_model_data = model_data
else:
    keep = list(dict.fromkeys(selected_features + ["review_text", "movie_id", "is_spoiler"]))
    full_model_data = load_training(columns=keep)
print(f"Full-training retuning: {len(full_model_data):,} reviews | features: {selected_features}")

summary, fold_scores, best_config = tune_linear_svm(
    full_model_data, model_grid=full_model_grid, tfidf_grid=tfidf_grid,
    feature_columns=selected_features, k=k, group_by=group_by, random_state=RANDOM_STATE,
    imbalance=imbalance, remove_spoilers=remove_spoilers, selection_metric=selection_metric,
    max_iter=max_iter, tol=tol,
)
if best_config is None:
    raise RuntimeError("No retuned candidate converged in every fold. Increase max_iter and rerun.")
display(summary[[c for c in show_columns if c in summary]])
print("Final configuration:", best_config)

first = baseline_summary[baseline_summary["converged"]].iloc[0]
last = summary[summary["converged"]].iloc[0]
picked = elimination_path.loc[elimination_path["selected"]].iloc[0]
comparison = pd.DataFrame([
    {"stage": "All features, tuned", "n_blocks": len(initial_features), "data": "sample",
     "rows": len(model_data), "score_mean": first[f"{selection_metric}_mean"],
     "score_std": first[f"{selection_metric}_std"]},
    {"stage": "Selected features, fixed settings", "n_blocks": len(selected_features),
     "data": "sample", "rows": len(model_data), "score_mean": picked["score_mean"],
     "score_std": picked["score_std"]},
    {"stage": "Selected features, retuned", "n_blocks": len(selected_features),
     "data": "full training", "rows": len(full_model_data),
     "score_mean": last[f"{selection_metric}_mean"], "score_std": last[f"{selection_metric}_std"]},
])
display(comparison)

## 9. Refit, Save and Interpret the Final Model

Fit fresh encoders and SVM weights on every training row with the final configuration. CV only
chose the *settings*; the fold models are thrown away, because each fold had its own vocabulary.

Saved under `data/models/<experiment>_full_backward/`: the fitted pipeline, all settings, both
CV tables, the elimination path/trials, the plots and the training-row indices.

**Reading the weights.** Positive weights push a review towards "spoiler", negative towards
"non-spoiler". The block-level table sums absolute weights per input, giving a rough idea of how
much the model relies on each input. Weights are comparable across blocks because every block is
scaled to a similar range (normalised TF-IDF rows, 0/1 genres, max-abs-scaled numbers).
Things to look for: warning words ("spoiler", "warning") → RQ4 shortcut; character names →
movie-specific memorisation (RQ3); usernames/blog signatures → reviewer shortcut.

In [ ]:
model, fit_info = refit_linear_svm(
    full_model_data, best_config, feature_columns=selected_features, random_state=RANDOM_STATE,
    imbalance=imbalance, remove_spoilers=remove_spoilers, max_iter=max_iter, tol=tol,
)
print("Final fit:", {key: round(value, 4) if isinstance(value, float) else value
                     for key, value in fit_info.items()})

output_dir = PROJECT_ROOT / "data/models" / f"{experiment}_full_backward"
output_dir.mkdir(parents=True, exist_ok=True)
baseline_summary.to_csv(output_dir / "baseline_cv_summary.csv")
baseline_fold_scores.to_csv(output_dir / "baseline_cv_fold_scores.csv", index=False)
elimination_path.to_json(output_dir / "elimination_path.json", orient="records", indent=2)
elimination_trials.to_json(output_dir / "elimination_trials.json", orient="records", indent=2)
summary.to_csv(output_dir / "cv_summary.csv")
fold_scores.to_csv(output_dir / "cv_fold_scores.csv", index=False)
fig_c.savefig(output_dir / "c_curve_and_margin.png", dpi=150, bbox_inches="tight")
fig_elim.savefig(output_dir / "backward_elimination.png", dpi=150, bbox_inches="tight")
if run_kernel_check:
    kernel_results.to_csv(output_dir / "kernel_check.csv", index=False)
    fig_kernel.savefig(output_dir / "kernel_check.png", dpi=150, bbox_inches="tight")
np.save(output_dir / "training_row_indices.npy", np.arange(len(full_model_data)))
np.save(output_dir / "selection_row_indices.npy", source_row_indices)
joblib.dump({
    "pipeline": model, "best_config": best_config, "feature_columns": selected_features,
    "fit_info": fit_info, "training_path": str(train_path),
    "training_rows": len(full_model_data), "feature_selection_rows": len(model_data),
    "total_training_rows": total_training_rows, "k": k, "group_by": group_by,
    "remove_spoilers": remove_spoilers, "imbalance": imbalance,
    "selection_metric": selection_metric, "decision_threshold": 0.0,
    "initial_features": initial_features, "baseline_config": baseline_config,
    "max_drop": max_drop, "model_grid": model_grid, "full_model_grid": full_model_grid,
    "tfidf_grid": tfidf_grid, "max_iter": max_iter, "tol": tol,
    "random_state": RANDOM_STATE, "sklearn_version": sklearn.__version__,
}, output_dir / "model.joblib", compress=3)
print("Saved model and CV results:", output_dir)

weights = top_features(model, n=20)
weights.to_csv(output_dir / "top_features.csv", index=False)
display(block_importance(model))
fig_w, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (direction, rows), color in zip(axes, weights.groupby("direction", sort=False),
                                        ["tab:red", "tab:blue"]):
    rows = rows.iloc[::-1]
    ax.barh(rows["feature"], rows["weight"], color=color)
    ax.set(title=f"Top {direction} features", xlabel="SVM weight")
fig_w.tight_layout()
fig_w.savefig(output_dir / "top_features.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Held-Out Test Evaluation (run once)

Run only after the features, settings and threshold are frozen. This section reloads the saved
pipeline and the matching test split and processes it in chunks. Nothing is fitted, oversampled,
selected or tuned here. If `remove_spoilers` is on, the pipeline removes the warnings from test
text itself, using the same rules as in training.

A review is predicted as a spoiler when its SVM decision value is **> 0** (the side of the
boundary it falls on). Average precision uses the decision values as ranking scores.
Test scores must **not** be used to change anything above.

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay
from src.clean_spoilers import clean_text
from src.metrics import evaluate

evaluation_dir = PROJECT_ROOT / "data/models" / f"{experiment}_full_backward"
saved = joblib.load(evaluation_dir / "model.joblib")
test_model = saved["pipeline"]
test_path = Path(saved["training_path"]).with_name("IMDB_reviews_test.json")
test_movies = pd.read_json(PROJECT_ROOT / "data/cleaned/IMDB_movie_details.json",
                           lines=True, convert_dates=False)

prediction_chunks = []
with pd.read_json(test_path, lines=True, convert_dates=False, chunksize=10000) as reader:
    for test_reviews in reader:
        test_features = prepare_features(test_reviews, test_movies)[saved["feature_columns"]]
        decision = test_model.decision_function(test_features)
        prediction_chunks.append(pd.DataFrame({
            "test_row": test_reviews.index.to_numpy(),
            "movie_id": test_reviews["movie_id"].to_numpy(),
            "is_spoiler": test_reviews["is_spoiler"].to_numpy().astype(int),
            "predicted_spoiler": (decision > saved["decision_threshold"]).astype(int),
            "decision_value": decision,
            "has_warning_phrase": [clean_text(t) != t for t in test_reviews["review_text"]],
            "review_excerpt": test_reviews["review_text"].str.slice(0, 300).to_numpy(),
        }))
test_predictions = pd.concat(prediction_chunks, ignore_index=True)
test_scores = evaluate(test_predictions["is_spoiler"], test_predictions["predicted_spoiler"],
                       test_predictions["decision_value"])
test_scores["average_precision"] = test_scores.pop("pr_auc")
metric_order = ["macro_f1", "precision", "recall", "f1", "average_precision", "accuracy"]
test_metrics = pd.DataFrame([test_scores])
test_confusion = pd.DataFrame(
    [[test_scores["tn"], test_scores["fp"]], [test_scores["fn"], test_scores["tp"]]],
    index=["Actual non-spoiler", "Actual spoiler"],
    columns=["Predicted non-spoiler", "Predicted spoiler"],
)
print(f"Held-out test reviews: {len(test_predictions):,}")
display(test_metrics[metric_order].T.rename(columns={0: "Test score"}))
display(test_confusion)

fig_t, axes = plt.subplots(1, 2, figsize=(13, 4.5))
PrecisionRecallDisplay.from_predictions(test_predictions["is_spoiler"],
                                        test_predictions["decision_value"], ax=axes[0],
                                        name="Linear SVM")
axes[0].axhline(test_predictions["is_spoiler"].mean(), color="grey", linestyle=":",
                label="Majority baseline (AP = spoiler rate)")
axes[0].legend()
axes[0].set_title("Precision-recall curve (test)")
for label, name in [(0, "Non-spoiler"), (1, "Spoiler")]:
    axes[1].hist(test_predictions.loc[test_predictions["is_spoiler"] == label, "decision_value"],
                 bins=80, alpha=0.55, density=True, label=name)
axes[1].axvline(0, color="black", linestyle="--", label="Decision boundary (0)")
for edge in (-1, 1):
    axes[1].axvline(edge, color="grey", linestyle=":")
axes[1].set(xlabel="SVM decision value  w·x + b", ylabel="Density",
            title="Where test reviews fall (dotted lines = margin)")
axes[1].legend()
fig_t.tight_layout()
plt.show()

test_predictions.to_csv(evaluation_dir / "test_predictions.csv", index=False)
test_metrics.to_csv(evaluation_dir / "test_metrics.csv", index=False)
test_confusion.to_csv(evaluation_dir / "test_confusion_matrix.csv")
fig_t.savefig(evaluation_dir / "test_pr_and_margin.png", dpi=150, bbox_inches="tight")

# Small committed summary row for the team comparison table (outputs/ is tracked by git).
results_row = {"model": experiment, "split": group_by, "features": "+".join(saved["feature_columns"]),
               **{key: saved["best_config"]["model"][key] for key in ("penalty", "loss", "C")},
               "ngram_range": str(saved["best_config"]["tfidf"]["ngram_range"]),
               **{m: round(float(test_scores[m]), 4) for m in metric_order},
               **{c: int(test_scores[c]) for c in ("tn", "fp", "fn", "tp")}}
(PROJECT_ROOT / "outputs").mkdir(exist_ok=True)
pd.DataFrame([results_row]).to_csv(PROJECT_ROOT / "outputs" / f"{experiment}_test_results.csv",
                                   index=False)
print("Saved test predictions and metrics:", evaluation_dir)